## Informationsgesellschaft und Datensicherheit

[Video1](https://youtu.be/pAjwAYoq2bY) - [Video2](https://youtu.be/yOYWkSyEue8) - [Video3](https://youtu.be/LLVNyTeWbV8)

### Sicherheitsziele

Kryptografische Verfahren dienen dazu, die folgenden Sicherheitsziele zu erreichen:

| Sicherheitsziel | Bedeutung | Wer greift an? |
|:--|:--|:--|
| **Vertraulichkeit** | Niemand außer dem Empfänger kann die Nachricht lesen. | Eve liest mit. |
| **Integrität** | Die Nachricht ist unterwegs nicht verändert worden. | Mallory verändert die Nachricht. |
| **Authentizität** | Die Nachricht stammt wirklich von dem angegebenen Absender. | Mallory gibt sich als Alice aus. |
| **Verbindlichkeit** | Der Absender kann später nicht abstreiten, die Nachricht geschickt zu haben. | Alice selbst will ihre Nachricht abstreiten. |

In diesem Kapitel lernen wir, mit welchen Methoden diese Ziele erreicht werden:

| Sicherheitsziel | erreicht durch |
|:--|:--|
| Vertraulichkeit | Verschlüsselung (symmetrisch, asymmetrisch, hybrid) |
| Integrität | Hashfunktion zusammen mit einer digitalen Signatur |
| Authentizität | digitale Signatur, Zertifikate (Public-Key-Infrastruktur) |
| Verbindlichkeit | digitale Signatur |

### Symmetrische Verschlüsselung

Bei einem **symmetrischen** Verfahren benutzen Sender und Empfänger **denselben geheimen Schlüssel** zum Ver- und Entschlüsseln. Alle Verfahren aus dem ersten Teil (Skytale, Cäsar, Vigenère, One-Time-Pad) sind symmetrisch.

Das wichtigste symmetrische Verfahren in der Praxis ist **AES** (Advanced Encryption Standard). Es wird zum Beispiel bei der Festplattenverschlüsselung, im WLAN (WPA2/WPA3) und bei HTTPS für die eigentlichen Nutzdaten eingesetzt.

Symmetrische Verfahren sind **schnell** und für große Datenmengen gut geeignet. Sie haben aber zwei Probleme:

1. **Schlüsseltausch:** Der gemeinsame Schlüssel muss vor der Kommunikation auf einem **sicheren Weg** ausgetauscht werden. Wer den Schlüssel unterwegs abfängt, kann alles mitlesen.
2. **Anzahl der Schlüssel:** Sollen sich *n* Personen paarweise vertraulich austauschen, braucht jedes Paar einen eigenen Schlüssel.

In [ ]:
def anzahl_schluessel(n):
    '''Anzahl der Schlüssel, damit sich n Personen paarweise austauschen können.'''
    return n * (n - 1) // 2

for n in [2, 5, 10, 100, 1000]:
    print(f'{n:5d} Personen -> {anzahl_schluessel(n):7d} symmetrische Schlüssel')

Jede der *n* Personen braucht einen Schlüssel für jede der *n − 1* anderen Personen. Dabei wird jeder Schlüssel doppelt gezählt (A–B und B–A), daher $\frac{n \cdot (n-1)}{2}$.

Bei 1000 Personen sind das fast eine halbe Million Schlüssel, die alle geheim erzeugt, verteilt und verwaltet werden müssen. Die **asymmetrische Verschlüsselung** löst diese Probleme.

### Asymmetrische Verschlüsselung

Jede Person besitzt ein **Schlüsselpaar**:

- **Öffentlicher Schlüssel** (*public key*): darf frei veröffentlicht werden. Mit ihm wird **verschlüsselt**.
- **Privater Schlüssel** (*private key*): bleibt geheim. Nur mit ihm kann man **entschlüsseln**.

```
Alice möchte Bob schreiben:
  Alice verschlüsselt mit  Bobs öffentlichem Schlüssel
  Bob   entschlüsselt mit  seinem privaten Schlüssel
```

Man kann sich das wie eine Adresse mit einem **Briefkasten** vorstellen: Jeder kann an der öffentlichen bekannten Adresse einen Brief einwerfen (öffentlicher Schlüssel), aber nur der Besitzer kann den Briefkasten mit seinem Schlüssel öffnen (privater Schlüssel).

Ein Lauscher kennt zwar Bobs öffentlichen Schlüssel, kann damit aber **nicht entschlüsseln**. Der öffentliche Schlüssel darf deshalb offen über das Internet übertragen werden. **Ein geheimer Schlüsseltausch entfällt.**

### Vergleich

| Kriterium | symmetrisch (z. B. AES) | asymmetrisch (z. B. RSA) |
|---|---|---|
| **Schlüssel** | ein gemeinsamer geheimer Schlüssel | Schlüsselpaar: öffentlich + privat |
| **Schlüsselverwaltung** | bei *n* Teilnehmern bis zu *n·(n−1)/2* Schlüssel | pro Person genau ein Schlüsselpaar, also *n* Paare |
| **Schlüsseltausch** | geheimer Kanal nötig, kritisch | öffentlicher Schlüssel wird offen verteilt |
| **Geschwindigkeit** | sehr schnell, auch für große Datenmengen | rechenaufwendig, etwa 100- bis 1000-mal langsamer |
| **typischer Einsatz** | Verschlüsselung der eigentlichen Nutzdaten | Schlüsselaustausch, Signaturen |

### Hybride Verschlüsselung

In der Praxis (HTTPS, Messenger, E-Mail-Verschlüsselung) werden beide Ansätze kombiniert:

1. Es wird ein **zufälliger symmetrischer Sitzungsschlüssel** erzeugt.
2. Dieser kurze Schlüssel wird **asymmetrisch** verschlüsselt übertragen (einfacher Schlüsseltausch).
3. Die eigentliche, oft große Nachricht wird **symmetrisch** mit dem Sitzungsschlüssel verschlüsselt (hohe Geschwindigkeit).

So verbindet man den einfachen Schlüsseltausch der asymmetrischen Verfahren mit der Geschwindigkeit der symmetrischen.

### Einwegfunktionen

Eine **Einwegfunktion** ist eine Funktion, die in eine Richtung leicht zu berechnen ist, deren Umkehrung aber praktisch nicht in vertretbarer Zeit gelingt.
Asymmetrische Verfahren beruhen auf solchen Funktionen.

Beim bekanntesten Verfahren, **RSA**, ist das die Multiplikation zweier großer Primzahlen:

| leichte Richtung | schwere Richtung |
|:--|:--|
| zwei Primzahlen multiplizieren: $p \cdot q = m$ | $m$ in die Primfaktoren $p$ und $q$ zerlegen |

Das Produkt $m$ steckt im öffentlichen Schlüssel. Wer $m$ zerlegen könnte, könnte auch den privaten Schlüssel berechnen.

Wir probieren das aus. Multiplizieren geht sofort, beim Zerlegen testen wir alle möglichen Teiler. Die Rechenzeit wird mit jeweils zwei Stellen mehr etwa zehnmal so lang.

In [ ]:
import time

def kleinster_teiler(m):
    t = 2
    while t * t <= m:
        if m % t == 0:
            return t
        t += 1
    return m

for p, q in [(1009, 1013), (10007, 10009), (100003, 100019), (1000003, 1000033), (10000019, 10000079)]:
    m = p * q
    start = time.perf_counter()
    t = kleinster_teiler(m)
    zeit = time.perf_counter() - start
    print(len(str(m)), 'Stellen:', m, '=', t, '*', m // t, round(zeit, 4), 's')

Die Laufzeit dieses Angriffs wächst **exponentiell** mit der Stellenzahl. In der Praxis benutzt man Zahlen mit mehr als 600 Dezimalstellen. Auch die besten bekannten Angriffsverfahren brauchen
dafür unvorstellbar lange. Deshalb gilt das Verfahren als sicher. Bewiesen ist das aber nicht: Es könnte ein schnelleres Verfahren geben, das nur noch niemand gefunden hat.

### Kryptographische Hashfunktionen

Eine **Hashfunktion** bildet eine beliebig lange Eingabe (einen Text, eine Datei) auf einen Wert **fester Länge** ab, den **Hashwert** der Eingabe. SHA-256 zum Beispiel liefert immer 256 Bit.

Eine **kryptographische** Hashfunktion muss zusätzlich folgende Eigenschaften haben:

- **Einwegfunktion:** Aus dem Hashwert lässt sich die Eingabe praktisch nicht rekonstruieren.
- **Kollisionsresistent:** Es ist praktisch unmöglich, zwei verschiedene Eingaben mit demselben Hashwert zu finden.
- **Lawineneffekt:** Schon eine winzige Änderung der Eingabe ergibt einen völlig anderen Hashwert.

Der Hashwert ist wie ein **Fingerabdruck** der Eingabe.

### Digitale Signatur

Mit einem asymmetrischen Verfahren und einer kryptographischen Hashfunktion kann Alice eine Nachricht **digital unterschreiben**. Dabei werden die Schlüssel umgekehrt benutzt wie beim Verschlüsseln: Alice arbeitet mit ihrem **privaten** Schlüssel, jeder kann mit ihrem **öffentlichen** Schlüssel prüfen.

<img src='./signatur.png'>

Im Bild steht $N$ für die Nachricht und *sig* für die Signatur.

**Signieren (Alice):** Alice berechnet den Hashwert ihrer Nachricht und verschlüsselt ihn mit ihrem **privaten** Schlüssel. Das ist die Signatur. Sie verschickt die Nachricht zusammen mit der Signatur.

**Prüfen (Bob):** Bob entschlüsselt die Signatur mit Alices **öffentlichem** Schlüssel. Das ergibt den Hashwert, den Alice gebildet hat. Bob berechnet selbst den Hashwert der empfangenen Nachricht und vergleicht. Stimmen beide überein, ist die Signatur gültig.

**Welche Sicherheitsziele erreicht die Signatur?**

- **Integrität:** Wird die Nachricht unterwegs verändert, passt der neu berechnete Hashwert nicht mehr zur Signatur.
- **Authentizität:** Eine gültige Signatur kann nur erzeugen, wer Alices privaten Schlüssel besitzt.
- **Verbindlichkeit:** Aus demselben Grund kann Alice später nicht abstreiten, die Nachricht signiert zu haben.
- **Vertraulichkeit** erreicht die Signatur **nicht**: Die Nachricht selbst wird unverschlüsselt verschickt. Soll sie geheim bleiben, muss sie zusätzlich verschlüsselt werden.

Alle diese Prüfungen setzen voraus, dass Bob den **richtigen** öffentlichen Schlüssel von Alice hat.

### Public-Key-Infrastruktur (PKI)

Ein Problem asymmetrischer Verfahren ist die **Authentizität des öffentlichen Schlüssels**: Gehört der öffentliche Schlüssel wirklich zu der Person oder Webseite, die ihn präsentiert? Ein Angreifer könnte auch seinen eigenen öffentlichen Schlüssel als den von Alice ausgeben.

Deshalb lässt Alice ihren öffentlichen Schlüssel von einer anerkannten Stelle bestätigen, einem **Trust-Center** (auch Zertifizierungsstelle, englisch *Certificate Authority*, CA):

<img src='./zertifizierung.png'>

Im Bild bedeutet „verschlüsselt mit $(m_V, d_V)$": Das Trust-Center **signiert** mit seinem privaten Schlüssel.

**Ablauf:**

1. Das Trust-Center prüft die Identität der Antragstellerin, zum Beispiel mit dem Personalausweis.
2. Es erstellt ein **Zertifikat**, das den öffentlichen Schlüssel und die Daten der Inhaberin enthält.
3. Es **signiert** das Zertifikat mit seinem eigenen privaten Schlüssel.
4. Wer den öffentlichen Schlüssel des Trust-Centers kennt und ihm vertraut, kann die Signatur prüfen und sich so von der Echtheit des enthaltenen Schlüssels überzeugen.

**Woher kennt Bob den öffentlichen Schlüssel des Trust-Centers?** Browser und Betriebssysteme bringen eine Liste vertrauenswürdiger **Wurzelzertifikate** mit. Oft ist ein Zertifikat von einer Zwischenstelle signiert, deren Zertifikat wieder von einer Wurzelstelle signiert ist. So entsteht eine **Zertifikatskette** (hierarchisches Vertrauensmodell).

##### Anwendungsbereiche

| Anwendung | Wozu dient das Zertifikat? |
|:--|:--|
| **Transportverschlüsselung** (HTTPS, TLS) | Der Browser prüft, ob der Server wirklich zur aufgerufenen Adresse gehört (Schloss-Symbol). |
| **Mailverschlüsselung** (S/MIME) | E-Mails werden signiert und verschlüsselt, das Zertifikat bestätigt den Absender. |
| **Digitaler Personalausweis** (eID) | Ausweis und Behörden-Server weisen sich gegenseitig mit Zertifikaten aus. |

##### Sicherheitsaspekte

- **Das ganze System hängt am Vertrauen in die Trust-Center.** Gerät der private Schlüssel eines Trust-Centers in falsche Hände, kann der Angreifer gefälschte Zertifikate ausstellen und sich als beliebige Stelle ausgeben. Dann müssen alle Zertifikate dieses Trust-Centers zurückgezogen werden. Das ist 2011 bei der niederländischen Zertifizierungsstelle DigiNotar passiert.
- **Ein Zertifikat bestätigt nur die Identität, nicht die Seriosität.** Auch eine Phishing-Seite kann ein gültiges Zertifikat für ihre eigene (ähnlich klingende) Adresse haben.
- **Zertifikate laufen ab und können gesperrt werden.** Programme müssen das prüfen.
- **Nutzerinnen und Nutzer** klicken Zertifikatswarnungen im Browser oft einfach weg und hebeln den Schutz damit aus.

### Datensicherheit und Datenschutz

**Datensicherheit** meint den Schutz von Daten vor Verlust, Manipulation und unbefugtem Zugriff, unabhängig davon, um welche Daten es sich handelt.

**Datenschutz** meint den Schutz des Rechts der betroffenen Personen auf **informationelle Selbstbestimmung**. Es dürfen nur die wirklich nötigen personenbezogenen Daten für einen festgelegten Zweck erhoben und verarbeitet werden. Grundsätze sind **Datensparsamkeit**, **Zweckbindung** und **Einwilligung**.

##### Beispiel

Eine Schule speichert Noten in einer Datenbank.

- **Datensicherheit:** Die Datenbank wird durch Passwörter geschützt und regelmäßig gesichert.
- **Datenschutz:** Es werden nur die nötigen Daten gespeichert, nur berechtigte Lehrkräfte dürfen die Noten einsehen, und die Daten werden nicht ohne Erlaubnis weitergegeben.

##### Maßnahmen zur Gewährleistung von Datensicherheit und Datenschutz

- **technisch:** verschlüsselte Speicherung und verschlüsselte Übertragung (HTTPS), Zugriffskontrolle mit persönlichen Konten, starke Passwörter und **Mehrfaktorauthentifizierung**, regelmäßige Sicherungskopien (Backups), aktuelle Software und Sicherheitsupdates
- **organisatorisch:** Rollen- und Rechtekonzept (jede Person sieht nur die für ihre Aufgabe nötigen Daten), Schulung der Mitarbeitenden, klare Regeln zur Weitergabe und Löschung von Daten, Protokollierung der Zugriffe

### Massenhafte Datenverarbeitung

Viele Dienste (zum Beispiel kostenlose Apps, soziale Netzwerke, Webshops) erheben und speichern dauerhaft große Mengen personenbezogener Daten und werten sie zentral aus. Typische Probleme für die Betroffenen:

- Aus scheinbar harmlosen Einzeldaten (zum Beispiel Standort, Nutzungszeiten) lassen sich durch **Zusammenführen** und Auswerten genaue Profile einzelner Personen erstellen (Wohnort, Arbeitsplatz, Tagesablauf, Gesundheitszustand).
- Eine spätere **Zweckänderung** oder Weitergabe an Dritte (Werbepartner, Versicherungen, Arbeitgeber) ist für die Betroffenen kaum kontrollierbar.
- Bei einem **Datenleck** können sensible Daten dauerhaft öffentlich werden.

Beim Geschäftsmodell „kostenlose App gegen Daten" wird nicht mit Geld, sondern mit persönlichen Daten bezahlt, deren Wert und spätere Verwendung die Nutzenden nicht überblicken.

##### Ein Szenario bewerten

Diese Fragen helfen bei der Bewertung einer massenhaften Datenverarbeitung:

1. Welche Daten werden erhoben, und welche davon sind für den Zweck wirklich nötig?
2. Zu welchem Zweck werden sie verarbeitet, und wer profitiert davon?
3. Haben die Betroffenen informiert eingewilligt, und können sie die Einwilligung zurückziehen?
4. Wer hat Zugriff auf die Daten, wo werden sie gespeichert, und wie lange?
5. Was kann passieren, wenn die Daten in falsche Hände geraten oder zu einem anderen Zweck genutzt werden?
6. Gibt es eine datensparsamere Alternative?